# Week 12: Memory and Knowledge Retrieval in Agents
## Complete Teaching Notebook — With Testing & Expected Outputs

**Course:** Professional Certificate Programme in Agentic AI and Applications

---

### Notebook Structure
Each Part has: **Concept** → **Build** → **Test** → **Expected Output** → **Key Takeaway**

| Part | What You Build | Testing Included |
|------|---------------|------------------|
| 1 | Memory Types (Buffer, Summary, Window) | Account number recall test |
| 2 | Token Cost Comparison | Token count comparison + quality matrix |
| 3 | FAISS Vector Store | Semantic search + cross-language test |
| 4 | RAG Pipeline | RAG vs bare LLM + hallucination detection |
| 5 | Memory Loop with Decay | Before/after decay comparison |
| 6 | Multi-Agent Shared Memory | Agent-to-agent retrieval verification |
| 7 | Customer Support with Memory | 3-call memory recall + customer separation |
| 8 | **End-to-End Integration Test** | Full pipeline validation |


---
## Setup — Run this cell first every session

```bash
pip install langchain-openai langchain faiss-cpu python-dotenv tiktoken langchain-community
```


In [ ]:
# !pip install langchain-openai langchain faiss-cpu python-dotenv tiktoken langchain-community

import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.schema import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()

api_ok = bool(os.getenv("OPENAI_API_KEY"))
print("=" * 55)
print("  Week 12 — Teaching Environment Ready")
print("=" * 55)
print(f"  LLM          : gpt-4o-mini (temperature=0)")
print(f"  Embeddings   : text-embedding-3-small (1536 dims)")
print(f"  API Key      : {'Loaded ✓' if api_ok else 'NOT FOUND ✗ — add .env file'}")
print(f"  Vector Store : FAISS (local, CPU, no signup)")
print("=" * 55)
assert api_ok, "API key not found. Create a .env file with OPENAI_API_KEY=sk-..."


**Expected Output:**
```
  Week 12 — Teaching Environment Ready
  LLM          : gpt-4o-mini (temperature=0)
  Embeddings   : text-embedding-3-small (1536 dims)
  API Key      : Loaded ✓
  Vector Store : FAISS (local, CPU, no signup)
```
If API Key shows `NOT FOUND` → create `.env` file in this folder.


---
# Part 1 — Memory Types in LangChain
### PPT: Block 2 | 12 minutes

| Memory Type | Stores | Token Cost | Recall |
|-------------|--------|------------|--------|
| BufferMemory | Every turn | HIGH | Complete |
| SummaryMemory | Compressed summary | LOW | Lossy |
| WindowMemory(k=3) | Last 3 turns only | FIXED | Recent only |


In [ ]:
from langchain.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
    ConversationBufferWindowMemory,
)
from langchain.chains import ConversationChain

buffer_mem  = ConversationBufferMemory()
summary_mem = ConversationSummaryMemory(llm=llm)
window_mem  = ConversationBufferWindowMemory(k=3)

chain_buffer  = ConversationChain(llm=llm, memory=buffer_mem,  verbose=False)
chain_summary = ConversationChain(llm=llm, memory=summary_mem, verbose=False)
chain_window  = ConversationChain(llm=llm, memory=window_mem,  verbose=False)

print("3 memory types created ✓")


In [ ]:
# 10-turn customer conversation
turns = [
    "Hi, my name is Sarah and my account number is AC-7829.",
    "I was charged twice for my subscription last month.",
    "The charge was $29.99 on March 5th and again on March 7th.",
    "I have been a customer for 3 years and this never happened before.",
    "Can you process a refund for the duplicate charge?",
    "How long will the refund take to appear?",
    "Will I get a confirmation email when it is processed?",
    "Also, can you make sure this does not happen next month?",
    "One more thing — can you update my email to sarah.new@email.com?",
    "Thanks. Can you summarise what we discussed today?",
]

for i, turn in enumerate(turns):
    chain_buffer.predict(input=turn)
    chain_summary.predict(input=turn)
    chain_window.predict(input=turn)
    print(f"  Turn {i+1}/10 ✓")

print("\nAll 10 turns processed.")


In [ ]:
# ── TEST: What does each memory recall? ────────────────────
tests = [
    ("Account number (Turn 1)", "What is the customer's account number?", "AC-7829"),
    ("Charge amount (Turn 3)",  "How much was the duplicate charge?",     "29.99"),
    ("Email update (Turn 9)",   "What email did the customer want?",      "sarah.new"),
]

print("=" * 65)
print("  RECALL TEST — 3 facts from different turns")
print("=" * 65)
print(f"  {'Test':<28} {'Buffer':^10} {'Summary':^10} {'Window':^10}")
print("  " + "-" * 58)

for test_name, question, keyword in tests:
    row = f"  {test_name:<28}"
    for name, chain in [("Buffer", chain_buffer), ("Summary", chain_summary), ("Window", chain_window)]:
        answer = chain.predict(input=question)
        found = keyword.lower() in answer.lower()
        row += f" {'✓':^10}" if found else f" {'✗':^10}"
    print(row)

print("\n  ✓ = correctly recalled  |  ✗ = lost or hallucinated")


**Expected Output:**
```
  Test                         Buffer     Summary    Window
  ----------------------------------------------------------
  Account number (Turn 1)       ✓          ✗          ✗
  Charge amount (Turn 3)        ✓          ✓          ✗
  Email update (Turn 9)         ✓          ✓          ✓
```

**Key Insight:** Window(k=3) only recalls Turn 9 (within window). Summary may compress away the account number. Buffer keeps everything.


---
# Part 2 — Token Cost Comparison
### PPT: Block 3 | 12 minutes


In [ ]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o-mini")

buffer_mem2  = ConversationBufferMemory()
summary_mem2 = ConversationSummaryMemory(llm=llm)
window_mem2  = ConversationBufferWindowMemory(k=5)

chain_b2 = ConversationChain(llm=llm, memory=buffer_mem2,  verbose=False)
chain_s2 = ConversationChain(llm=llm, memory=summary_mem2, verbose=False)
chain_w2 = ConversationChain(llm=llm, memory=window_mem2,  verbose=False)

turns_15 = [
    "My name is James, account JM-4401.",
    "I ordered a laptop on March 1st, order #ORD-9921.",
    "It was supposed to arrive by March 5th but still has not come.",
    "Tracking says in transit for 5 days now.",
    "This is a work laptop — I am losing productivity every day.",
    "Can you check with the carrier for the actual delivery date?",
    "I need this resolved today or I want a full refund.",
    "What are my options? Reship or refund?",
    "I will take the reship if it arrives within 2 days.",
    "Can you expedite to overnight shipping?",
    "Also apply a 20 percent discount for the inconvenience.",
    "What is the tracking number for the new shipment?",
    "Change my delivery address to 456 Oak Avenue, Suite 12.",
    "Confirm: overnight shipping, 20 percent discount, new address.",
    "Thanks. What is your name so I can reference this conversation?",
]

for i, turn in enumerate(turns_15):
    chain_b2.predict(input=turn)
    chain_s2.predict(input=turn)
    chain_w2.predict(input=turn)

print("=" * 65)
print("  TOKEN COST COMPARISON — 15 turns")
print("=" * 65)
print(f"  {'Memory Type':<15} {'Tokens':>8} {'$/call':>10} {'$/month':>12}")
print("  " + "-" * 45)

for name, mem in [("Buffer", buffer_mem2), ("Summary", summary_mem2), ("Window(k=5)", window_mem2)]:
    content = mem.buffer if hasattr(mem, 'buffer') else str(mem.load_memory_variables({}))
    tokens = len(enc.encode(content))
    cost_call = tokens * 0.15 / 1_000_000
    cost_month = cost_call * 1000 * 30
    print(f"  {name:<15} {tokens:>8,} {cost_call:>10.5f} {cost_month:>12.2f}")


In [ ]:
# ── QUALITY TEST across all 3 memory types ─────────────────
test_q = [
    ("Account number?", "JM-4401"),
    ("Order number?", "ORD-9921"),
    ("Discount agreed?", "20"),
    ("New address?", "456 Oak"),
]

print("=" * 65)
print("  QUALITY MATRIX")
print("=" * 65)
print(f"  {'Question':<22} {'Buffer':^10} {'Summary':^10} {'Window':^10}")
print("  " + "-" * 52)

for question, keyword in test_q:
    row = f"  {question:<22}"
    for name, chain in [("Buffer", chain_b2), ("Summary", chain_s2), ("Window", chain_w2)]:
        answer = chain.predict(input=question)
        found = keyword.lower() in answer.lower()
        row += f" {'✓':^10}" if found else f" {'✗':^10}"
    print(row)


**Expected Output:**
```
  TOKEN COST COMPARISON — 15 turns
  Memory Type       Tokens     $/call     $/month
  Buffer            ~4,000    0.00060       18.00
  Summary             ~800    0.00012        3.60
  Window(k=5)         ~600    0.00009        2.70

  QUALITY MATRIX
  Question               Buffer     Summary    Window
  Account number?          ✓          ✗          ✗
  Order number?            ✓          ✓          ✗
  Discount agreed?         ✓          ✓          ✓
  New address?             ✓          ✓          ✓
```
Buffer = best quality, highest cost. Window = cheapest, loses early context. **Production answer: RAG (Part 4).**


---
# Part 3 — Build a FAISS Vector Store
### PPT: Block 4 | 12 minutes


In [ ]:
from langchain_community.vectorstores import FAISS

knowledge_base = [
    "Refund policy: Full refund within 30 days. Partial refund (50%) within 60 days.",
    "Shipping: Standard 3-5 days. Express 1-2 days for $9.99.",
    "Premium members get free express shipping on orders over $50.",
    "Password reset: Settings > Security > Reset Password. Verification email sent.",
    "Account deletion: Contact support. Data removed within 30 days (GDPR).",
    "Payment: Visa, MasterCard, Amex, PayPal. Apple Pay and Google Pay on mobile.",
    "Order tracking: Tracking link emailed after shipping. Allow 24 hours to activate.",
    "International shipping: 45 countries. 7-14 days. Import duties are customer responsibility.",
    "Gift returns: Recipients get store credit. Original purchaser gets refund.",
    "Damaged items: Report within 48 hours with photos. Free replacement in 2 days.",
    "Subscriptions: Basic $9.99/mo, Pro $19.99/mo, Enterprise custom. All include 24/7 support.",
    "Cancellation: Cancel anytime, no fee. Refund for unused portion of billing cycle.",
    "Price matching: Match competitor prices within 14 days with proof.",
    "Bulk orders: 50+ units = 15% discount. Contact sales@company.com.",
    "Warranty: Electronics have 2-year warranty. Extended warranty $29.99/year.",
    "Store hours: Online 24/7. Support Mon-Fri 8am-8pm EST, Sat 9am-5pm EST.",
    "Loyalty: 1 point per dollar. 100 points = $5. Points expire after 12 months inactivity.",
    "Size exchanges: Free within 30 days. Unworn with tags.",
    "Two-factor auth: Settings > Security > 2FA. SMS or authenticator app.",
    "Data privacy: Only necessary data collected. Export anytime. GDPR/CCPA compliant.",
]

store = FAISS.from_texts(knowledge_base, embeddings)
print(f"FAISS store: {store.index.ntotal} documents indexed, {store.index.d} dimensions ✓")


In [ ]:
# ── SEMANTIC SEARCH TEST ───────────────────────────────────
tests = [
    ("Can I get my money back?",       "Refund"),
    ("How do I change my password?",   "Password"),
    ("Do you deliver overseas?",       "International"),
    ("My item arrived broken",         "Damaged"),
    ("I want to stop paying",          "Cancel"),
]

print("=" * 65)
print("  SEMANTIC SEARCH TEST")
print("=" * 65)

passed = 0
for query, expected in tests:
    results = store.similarity_search_with_score(query, k=1)
    doc, score = results[0]
    matched = expected.lower() in doc.page_content.lower()
    passed += int(matched)
    print(f"  Q: '{query}'")
    print(f"     → [{score:.3f}] {'✓' if matched else '✗'} {doc.page_content[:60]}...")

print(f"\n  Result: {passed}/{len(tests)} queries matched correctly")


In [ ]:
# ── CROSS-LANGUAGE TEST ────────────────────────────────────
print("=" * 65)
print("  CROSS-LANGUAGE TEST — 'return a gift' in 5 languages")
print("=" * 65)

for lang, q in [
    ("English",  "Can I return a gift?"),
    ("Spanish",  "¿Puedo devolver un regalo?"),
    ("French",   "Puis-je retourner un cadeau?"),
    ("Hindi",    "क्या मैं उपहार वापस कर सकता हूं?"),
    ("German",   "Kann ich ein Geschenk zurückgeben?"),
]:
    results = store.similarity_search_with_score(q, k=1)
    doc, score = results[0]
    is_gift = "gift" in doc.page_content.lower()
    print(f"  [{lang:8}] [{score:.3f}] {'✓' if is_gift else '✗'} {doc.page_content[:50]}...")


**Expected:** All 5 languages find "Gift returns" doc. Embeddings encode **meaning** across languages.


---
# Part 4 — Build a RAG Pipeline
### PPT: Block 5 | 15 minutes

```
User Question → FAISS search → Top-3 docs → Inject into prompt → LLM answers
```


In [ ]:
from langchain.chains import RetrievalQA

retriever = store.as_retriever(search_kwargs={"k": 3})
rag_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)
print("RAG chain created ✓")


In [ ]:
# ── RAG ANSWERS WITH SOURCES ───────────────────────────────
questions = [
    "I bought something 45 days ago. Can I still get a refund?",
    "How do I set up two-factor authentication?",
    "My laptop screen was cracked on arrival. What do I do?",
    "Can you match a price I found on Amazon?",
    "How many loyalty points do I need for a $5 discount?",
]

print("=" * 65)
print("  RAG PIPELINE — Grounded Answers")
print("=" * 65)

for q in questions:
    result = rag_chain.invoke({"query": q})
    print(f"\n  Q: {q}")
    print(f"  A: {result['result'][:150]}...")
    print(f"  Sources: {[d.page_content[:40]+'...' for d in result['source_documents']]}")


In [ ]:
# ── HALLUCINATION TEST: RAG vs Bare LLM ───────────────────
q = "What is your refund policy for items returned after 30 days?"

rag_result = rag_chain.invoke({"query": q})
bare_result = llm.invoke([HumanMessage(content=q)]).content

print("=" * 65)
print("  HALLUCINATION TEST")
print("=" * 65)
print(f"\n  Q: {q}\n")
print(f"  RAG:  {rag_result['result'][:200]}...")
print(f"  Bare: {bare_result[:200]}...")

has_50 = "50" in rag_result['result']
print(f"\n  RAG mentions 50% partial refund: {'✓ GROUNDED' if has_50 else 'check manually'}")
print(f"  Bare LLM may invent a policy. The RAG answer has source docs to verify.")


**Expected:** RAG cites "50% partial refund" (from knowledge base). Bare LLM may hallucinate a plausible but wrong policy.


---
# Part 5 — Memory Loop with Decay
### PPT: Block 7 | 15 minutes

Beyond RAG: Query → Retrieve → Generate → **STORE** → **DECAY**


In [ ]:
class MemoryLoopAgent:
    def __init__(self, name, knowledge_texts):
        self.name = name
        self.store = FAISS.from_texts(knowledge_texts, embeddings)
        self.interactions = []
        self.initial_doc_count = self.store.index.ntotal

    def respond(self, query):
        retrieved = self.store.similarity_search_with_score(query, k=3)
        context = "\n".join([doc.page_content for doc, _ in retrieved])

        response = llm.invoke([
            SystemMessage(content=f"You are {self.name}. Use ONLY this context:\n{context}"),
            HumanMessage(content=query)
        ]).content

        record = f"Customer asked: {query} | Agent answered: {response[:120]}"
        self.store.add_texts([record])
        self.interactions.append({"query": query, "response": response[:120]})
        return response, retrieved

    def get_stats(self):
        return {
            "total_vectors": self.store.index.ntotal,
            "original_docs": self.initial_doc_count,
            "stored_interactions": len(self.interactions),
        }

print("MemoryLoopAgent defined ✓")


In [ ]:
agent = MemoryLoopAgent("SupportBot", knowledge_base)

queries = [
    ("Q1", "What is the refund policy?"),
    ("Q2", "How do I track my order?"),
    ("Q3", "I want to cancel my subscription."),
    ("Q4", "Do you offer a student discount?"),
    ("Q5", "What payment methods do you accept?"),
    ("Q6", "What did I ask you about refunds earlier?"),
]

print("=" * 60)
print("  MEMORY LOOP — Q6 tests recall of Q1")
print("=" * 60)

for label, q in queries:
    response, retrieved = agent.respond(q)
    print(f"\n  {label}: {q}")
    print(f"  Answer: {response[:100]}...")

stats = agent.get_stats()
print(f"\n  Stats: {stats['total_vectors']} vectors ({stats['original_docs']} docs + {stats['stored_interactions']} interactions)")


In [ ]:
# ── TEST: Does Q6 retrieve the stored Q1 interaction? ──────
print("=" * 60)
print("  MEMORY RECALL TEST")
print("=" * 60)

results = agent.store.similarity_search_with_score("What did I ask about refunds?", k=5)
found = False
for doc, score in results:
    is_interaction = "Customer asked:" in doc.page_content
    if is_interaction: found = True
    marker = "← STORED" if is_interaction else "← ORIGINAL"
    print(f"  [{score:.3f}] {marker} | {doc.page_content[:70]}...")

print(f"\n  Memory recall: {'✓ PASS' if found else '✗ FAIL'}")


**Expected:** Q6 retrieves the stored Q1 interaction. The agent found its own prior answer. That IS the memory loop.


---
# Part 6 — Multi-Agent Shared Memory
### PPT: Block 8 | 12 minutes

Two agents, one FAISS store. Researcher writes → Analyst reads.


In [ ]:
shared_store = FAISS.from_texts([
    "Policy: refunds over $100 require manager approval.",
    "Escalation: if customer mentions lawyer, escalate immediately.",
    "VIP customers (prefix VIP-) get 48-hour resolution guarantee.",
], embeddings)

def researcher_agent(query):
    response = llm.invoke([
        SystemMessage(content="Research agent. Write a 2-sentence finding."),
        HumanMessage(content=query)
    ]).content
    shared_store.add_texts([f"RESEARCH FINDING: {response}"])
    return response

def analyst_agent(query):
    results = shared_store.similarity_search(query, k=3)
    context = "\n".join([d.page_content for d in results])
    response = llm.invoke([
        SystemMessage(content=f"Analyst. Decide based on:\n{context}"),
        HumanMessage(content=query)
    ]).content
    return response, results

print(f"Shared store: {shared_store.index.ntotal} policy docs ✓")


In [ ]:
print("=" * 60)
print("  MULTI-AGENT SHARED MEMORY TEST")
print("=" * 60)

print(f"\n  Store before: {shared_store.index.ntotal} vectors")

finding = researcher_agent("Customer VIP-3301 charged $150 twice, threatening legal action.")
print(f"\n  Researcher wrote: {finding[:100]}...")
print(f"  Store after: {shared_store.index.ntotal} vectors (+1)")

decision, retrieved = analyst_agent("Handle VIP-3301 duplicate charge?")
print(f"\n  Analyst decision: {decision[:200]}...")

found_research = any("RESEARCH FINDING" in d.page_content for d in retrieved)
found_vip = any("VIP" in d.page_content for d in retrieved)
found_escalate = any("escalat" in d.page_content.lower() for d in retrieved)

print(f"\n  Analyst found researcher's finding: {'✓' if found_research else '✗'}")
print(f"  Analyst found VIP policy:           {'✓' if found_vip else '✗'}")
print(f"  Analyst found escalation rule:      {'✓' if found_escalate else '✗'}")


**Expected:** Analyst retrieves researcher's finding + VIP policy + escalation rule — **without direct communication**. Shared memory = indirect coordination.


---
# Part 7 — Customer Support Agent with Memory (Capstone)
### PPT: Block 9 | 15 minutes

Dual stores: Knowledge Base (policies) + Interaction Store (customer history).


In [ ]:
class MemoryCustomerSupportAgent:
    def __init__(self, knowledge_texts):
        self.kb_store = FAISS.from_texts(knowledge_texts, embeddings)
        self.interaction_store = None
        self.call_log = []

    def _add_interaction(self, text):
        if self.interaction_store is None:
            self.interaction_store = FAISS.from_texts([text], embeddings)
        else:
            self.interaction_store.add_texts([text])

    def handle_call(self, customer_name, message):
        kb_results = self.kb_store.similarity_search(message, k=2)
        kb_ctx = "\n".join([d.page_content for d in kb_results])

        hist_ctx = "(No prior interactions)"
        if self.interaction_store:
            hist = self.interaction_store.similarity_search(f"{customer_name}: {message}", k=3)
            if hist:
                hist_ctx = "\n".join([d.page_content for d in hist])

        prompt = f'''You are a helpful support agent.

Company Policies:
{kb_ctx}

Previous Interactions:
{hist_ctx}

Rules:
- Reference prior history naturally if it exists
- Use policies for accurate answers
- Be concise'''

        response = llm.invoke([
            SystemMessage(content=prompt),
            HumanMessage(content=f"Customer {customer_name}: {message}")
        ]).content

        record = f"[{customer_name}] Said: {message} | Agent: {response[:150]}"
        self._add_interaction(record)
        self.call_log.append({"customer": customer_name, "message": message, "response": response[:150]})
        return response

print("MemoryCustomerSupportAgent defined ✓")


In [ ]:
support = MemoryCustomerSupportAgent(knowledge_base)

calls = [
    ("Sarah Miller", "I was charged twice — $29.99 on March 5th and 7th. Can I get a refund?"),
    ("Sarah Miller", "Hi, I called yesterday about billing. Has my refund been processed?"),
    ("Sarah Miller", "5 days and no refund. What is going on?"),
]

for i, (name, msg) in enumerate(calls):
    print(f"\n{'='*60}")
    print(f"  CALL {i+1} — {name}")
    print(f"{'='*60}")
    print(f"  Customer: {msg}")
    r = support.handle_call(name, msg)
    print(f"  Agent: {r[:220]}...")
    if support.interaction_store:
        print(f"  [Store: {support.interaction_store.index.ntotal} interactions]")


In [ ]:
# ── TESTS ──────────────────────────────────────────────────
print("=" * 60)
print("  CUSTOMER SUPPORT MEMORY TESTS")
print("=" * 60)

# Test 1: Call 2 references Call 1
c2 = support.call_log[1]['response'].lower()
t1 = any(w in c2 for w in ['charged', 'billing', 'refund', '29.99', 'yesterday', 'previous'])
print(f"  Test 1 — Call 2 recalls Call 1: {'✓ PASS' if t1 else '✗ FAIL'}")

# Test 2: Call 3 shows timeline
c3 = support.call_log[2]['response'].lower()
t2 = any(w in c3 for w in ['refund', 'process', 'initiated', 'status'])
print(f"  Test 2 — Call 3 timeline aware: {'✓ PASS' if t2 else '✗ FAIL'}")

# Test 3: Different customer doesn't see Sarah's data
r_other = support.handle_call("James Wong", "I want to cancel.")
t3 = "sarah" not in r_other.lower() and "29.99" not in r_other.lower()
print(f"  Test 3 — James can't see Sarah: {'✓ PASS' if t3 else '✗ FAIL — data leaked!'}")

print(f"\n  Total interactions stored: {support.interaction_store.index.ntotal}")


**Expected:**
```
  Test 1 — Call 2 recalls Call 1: ✓ PASS
  Test 2 — Call 3 timeline aware: ✓ PASS
  Test 3 — James can't see Sarah: ✓ PASS
```


---
# Part 8 — End-to-End Integration Test
### Full pipeline validation across all components


In [ ]:
print("=" * 60)
print("  END-TO-END INTEGRATION TEST")
print("=" * 60)

passed = failed = 0

# 1. Memory types
print("\n  [1/6] Memory Types...")
try:
    m = ConversationBufferMemory()
    c = ConversationChain(llm=llm, memory=m, verbose=False)
    c.predict(input="Test"); passed += 1; print("        ✓")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 2. FAISS store
print("  [2/6] FAISS Vector Store...")
try:
    ts = FAISS.from_texts(["test doc"], embeddings)
    assert len(ts.similarity_search("test", k=1)) > 0
    passed += 1; print("        ✓")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 3. RAG
print("  [3/6] RAG Pipeline...")
try:
    tr = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff",
         retriever=store.as_retriever(search_kwargs={"k": 2}),
         return_source_documents=True)
    r = tr.invoke({"query": "refund"})
    assert 'result' in r and len(r['source_documents']) > 0
    passed += 1; print("        ✓")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 4. Memory loop
print("  [4/6] Memory Loop...")
try:
    la = MemoryLoopAgent("Test", ["test entry"])
    la.respond("question 1"); la.respond("question 2")
    assert la.get_stats()['stored_interactions'] == 2
    passed += 1; print("        ✓")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 5. Shared memory
print("  [5/6] Shared Memory...")
try:
    ss = FAISS.from_texts(["base doc"], embeddings)
    n = ss.index.ntotal
    ss.add_texts(["RESEARCH FINDING: test"])
    assert ss.index.ntotal == n + 1
    r = ss.similarity_search("research", k=1)
    assert "RESEARCH" in r[0].page_content
    passed += 1; print("        ✓")
except Exception as e: failed += 1; print(f"        ✗ {e}")

# 6. Customer support
print("  [6/6] Customer Support...")
try:
    sa = MemoryCustomerSupportAgent(knowledge_base)
    sa.handle_call("Test", "refund"); sa.handle_call("Test", "follow up")
    assert sa.interaction_store.index.ntotal >= 2
    passed += 1; print("        ✓")
except Exception as e: failed += 1; print(f"        ✗ {e}")

print(f"\n  RESULTS: {passed}/{passed+failed} passed")
print(f"  {'✓ ALL TESTS PASSED' if failed == 0 else f'✗ {failed} FAILED'}")
print("=" * 60)


**Expected:**
```
  [1/6] Memory Types...      ✓
  [2/6] FAISS Vector Store.. ✓
  [3/6] RAG Pipeline...      ✓
  [4/6] Memory Loop...       ✓
  [5/6] Shared Memory...     ✓
  [6/6] Customer Support...  ✓

  RESULTS: 6/6 passed
  ✓ ALL TESTS PASSED
```

---

## Summary

| Part | Component | Key Pattern |
|------|-----------|-------------|
| 1 | Memory Types | Buffer / Summary / Window comparison |
| 2 | Token Cost | Quality vs cost trade-off matrix |
| 3 | Vector Store | FAISS: embed → index → search by meaning |
| 4 | RAG Pipeline | Retrieve → Augment → Generate (grounded answers) |
| 5 | Memory Loop | RAG + Store + Decay (agent improves over time) |
| 6 | Shared Memory | Multi-agent coordination through one store |
| 7 | Customer Support | Dual stores: KB + interaction history |
| 8 | Integration | End-to-end pipeline verification |

### 5 Laws of Production Agent Memory
1. **PERSISTENCE** — survives restarts
2. **RETRIEVAL** — searchable by meaning
3. **DECAY** — old memories fade
4. **SEPARATION** — shared ≠ private
5. **GOVERNANCE** — read/write/delete audit trail
